# PCA: see covariance, eigenvectors, and projection

Companion to HTB Academy Module 290, Section 11. Three invented two-feature rows make the eigen-equation and dimensionality reduction exact enough to inspect by hand. The two features already share the same scale, so this teaching example **centers without standardizing**; HTB recommends standardization when feature scales differ. This is not a fitted security pipeline.

Read $Cv=\lambda v$ as *C times v equals lambda times v*: covariance matrix C acts on direction v by scaling it by eigenvalue lambda. Read $Y=XV$ as *Y equals X times V*, where X here is **centered data** and V contains the retained eigenvector directions.

In [ ]:
import numpy as np

# Rows are observations; columns are features. Both features have equal units.
raw = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]])
feature_means = raw.mean(axis=0)  # Mean each column using the training rows.
centered = raw - feature_means  # X: subtract the same feature means from every row.
# Sample covariance C = X^T X / (n-1). X^T is X transposed.
covariance = centered.T @ centered / (len(raw) - 1)
print('means:', feature_means)
print('centered rows:\n', centered)
print('sample covariance:\n', covariance)
assert np.allclose(feature_means, [2, 2])
assert np.allclose(covariance, [[1, 1], [1, 1]])

## Solve the covariance eigen-equation

An eigenvector is a direction that covariance does not rotate; its eigenvalue reports variance along that unit direction. `np.linalg.eigh` is an eigensolver for symmetric matrices like covariance. It returns eigenvalues in ascending order, so we reverse that order. Eigenvector signs can flip with no change in the geometric axis.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
descending = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[descending]
eigenvectors = eigenvectors[:, descending]  # Columns are the directions.
first_direction = eigenvectors[:, 0]
print('eigenvalues (largest first):', eigenvalues)
print('first unit direction:', first_direction)
# Check C v = lambda v without assuming the solver's arbitrary sign for v.
assert np.allclose(covariance @ first_direction, eigenvalues[0] * first_direction)
assert np.allclose(eigenvalues, [2, 0])
assert np.allclose(np.abs(first_direction), [1 / np.sqrt(2)] * 2)

## Keep one direction and recover the toy data

Read $Y=XV$ as *project centered rows X onto selected directions V*. Here V has one column, so each two-feature row becomes one score. The explained-variance ratio is *its eigenvalue divided by the sum of all eigenvalues*. Reconstruction multiplies scores by the direction's transpose and adds the original means. Exact recovery happens only because the discarded direction has zero variance in these three rows.

In [ ]:
selected_direction = eigenvectors[:, :1]  # V has shape 2-by-1.
scores = centered @ selected_direction  # Y = X V, shape 3-by-1.
explained_ratio = eigenvalues[0] / eigenvalues.sum()
reconstructed = scores @ selected_direction.T + feature_means
print('one-component scores:', scores.ravel())
print(f'fraction of variance kept: {explained_ratio:.0%}')
print('reconstructed rows:\n', reconstructed)
assert scores.shape == (3, 1)
assert np.allclose(np.abs(scores.ravel()), [np.sqrt(2), 0, np.sqrt(2)])
assert explained_ratio == 1.0
assert np.allclose(reconstructed, raw)

## Security interpretation

Variance is not the same as relevance. A rare attack indicator might occupy a low-variance direction that PCA would discard. Fit means, scaling, and PCA directions on training data only; inspect downstream detection quality before trusting a reduced representation. PCA also does not make sensitive data private.